In [1]:
import findspark
findspark.init("/usr/local/spark")

In [2]:
from pyspark.sql import SparkSession, functions as F

In [3]:
spark = SparkSession.builder \
.appName("Spark YARN Example") \
.master("yarn") \
.enableHiveSupport() \
.getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/10/16 15:22:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/10/16 15:22:50 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


In [4]:
! curl -O https://raw.githubusercontent.com/erkansirin78/datasets/master/Advertising.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  4556  100  4556    0     0   7266      0 --:--:-- --:--:-- --:--:--  7254


In [8]:
! head Advertising.csv

ID,TV,Radio,Newspaper,Sales
1,230.1,37.8,69.2,22.1
2,44.5,39.3,45.1,10.4
3,17.2,45.9,69.3,9.3
4,151.5,41.3,58.5,18.5
5,180.8,10.8,58.4,12.9
6,8.7,48.9,75,7.2
7,57.5,32.8,23.5,11.8
8,120.2,19.6,11.6,13.2
9,8.6,2.1,1,4.8


In [5]:
! hdfs dfs -mkdir -p Advertising.csv /user/root/

In [11]:
! hdfs dfs -put Advertising.csv /user/root/

In [12]:
! hdfs dfs -head /user/root/Advertising.csv

ID,TV,Radio,Newspaper,Sales
1,230.1,37.8,69.2,22.1
2,44.5,39.3,45.1,10.4
3,17.2,45.9,69.3,9.3
4,151.5,41.3,58.5,18.5
5,180.8,10.8,58.4,12.9
6,8.7,48.9,75,7.2
7,57.5,32.8,23.5,11.8
8,120.2,19.6,11.6,13.2
9,8.6,2.1,1,4.8
10,199.8,2.6,21.2,10.6
11,66.1,5.8,24.2,8.6
12,214.7,24,4,17.4
13,23.8,35.1,65.9,9.2
14,97.5,7.6,7.2,9.7
15,204.1,32.9,46,19
16,195.4,47.7,52.9,22.4
17,67.8,36.6,114,12.5
18,281.4,39.6,55.8,24.4
19,69.2,20.5,18.3,11.3
20,147.3,23.9,19.1,14.6
21,218.4,27.7,53.4,18
22,237.4,5.1,23.5,12.5
23,13.2,15.9,49.6,5.6
24,228.3,16.9,26.2,15.5
25,62.3,12.6,18.3,9.7
26,262.9,3.5,19.5,12
27,142.9,29.3,12.6,15
28,240.1,16.7,22.9,15.9
29,248.8,27.1,22.9,18.9
30,70.6,16,40.8,10.5
31,292.9,28.3,43.2,21.4
32,112.9,17.4,38.6,11.9
33,97.2,1.5,30,9.6
34,265.6,20,0.3,17.4
35,95.7,1.4,7.4,9.5
36,290.7,4.1,8.5,12.8
37,266.9,43.8,5,25.4
38,74.7,49.4,45.7,14.7
39,43.1,26.7,35.1,10.1
40,228,37.7,32,21.5
41,202.5,22.3,31.6,16.6
42,177,33.4,38.7,17.1
43,293.6,27.7,1.8,20.7
44,206.9,8.4,26.4,12.9
45,25

In [13]:
df_hdfs = spark.read.format("csv").option("header", True) \
.option("inferSchema", True).option("sep",",") \
.load("hdfs://cluster-master:9000/user/root/Advertising.csv")

In [14]:
df_hdfs.show(5)

+---+-----+-----+---------+-----+
| ID|   TV|Radio|Newspaper|Sales|
+---+-----+-----+---------+-----+
|  1|230.1| 37.8|     69.2| 22.1|
|  2| 44.5| 39.3|     45.1| 10.4|
|  3| 17.2| 45.9|     69.3|  9.3|
|  4|151.5| 41.3|     58.5| 18.5|
|  5|180.8| 10.8|     58.4| 12.9|
+---+-----+-----+---------+-----+
only showing top 5 rows



In [15]:
spark.sql("show databases").show()

+---------+
|namespace|
+---------+
|  default|
+---------+



In [16]:
spark.sql("create database if not exists test1")

DataFrame[]

In [17]:
spark.sql("use test1")

DataFrame[]

In [18]:
spark.sql("show tables;").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
+---------+---------+-----------+



## Lowecase all columns strangely table contens can't be seen from hive.

In [24]:
df_hdfs = df_hdfs.toDF(*[c.lower() for c in df_hdfs.columns])

In [25]:
df_hdfs.show(3)

+---+-----+-----+---------+-----+
| id|   tv|radio|newspaper|sales|
+---+-----+-----+---------+-----+
|  1|230.1| 37.8|     69.2| 22.1|
|  2| 44.5| 39.3|     45.1| 10.4|
|  3| 17.2| 45.9|     69.3|  9.3|
+---+-----+-----+---------+-----+
only showing top 3 rows



In [26]:
df_hdfs.write.mode("overwrite").format("orc").saveAsTable("test1.spark_adv_orc")

In [27]:
spark.sql("show tables;").show()

+---------+-------------+-----------+
|namespace|    tableName|isTemporary|
+---------+-------------+-----------+
|    test1|spark_adv_orc|      false|
+---------+-------------+-----------+



In [28]:
df_hive = spark.table("test1.spark_adv_orc")

In [29]:
df_hive.show(4)

+---+-----+-----+---------+-----+
| id|   tv|radio|newspaper|sales|
+---+-----+-----+---------+-----+
|  1|230.1| 37.8|     69.2| 22.1|
|  2| 44.5| 39.3|     45.1| 10.4|
|  3| 17.2| 45.9|     69.3|  9.3|
|  4|151.5| 41.3|     58.5| 18.5|
+---+-----+-----+---------+-----+
only showing top 4 rows



In [31]:
spark.stop()